In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

# Busca la raíz del proyecto desde la ubicación del notebook
ROOT = Path.cwd()

while ROOT != ROOT.parent and not (ROOT / "database").exists():
    ROOT = ROOT.parent

DB_PATH = ROOT / "database" / "air_quality.db"

print("Ruta de la base de datos:", DB_PATH)
print("¿La base existe?:", DB_PATH.exists())

conexion = sqlite3.connect(DB_PATH)

Ruta de la base de datos: /workspaces/etl-calidad-aire-india/database/air_quality.db
¿La base existe?: True


In [2]:
consulta = """
SELECT
    f.measurement_id,
    c.city_name AS City,
    d.full_datetime AS Datetime,
    f.pm25 AS "PM2.5",
    f.pm10 AS PM10,
    f.no_value AS NO,
    f.no2 AS NO2,
    f.nox AS NOx,
    f.nh3 AS NH3,
    f.co AS CO,
    f.so2 AS SO2,
    f.o3 AS O3,
    f.aqi AS AQI,
    a.aqi_bucket AS AQI_Bucket,
    d.year AS Year,
    d.month AS Month,
    d.day AS Day,
    d.quarter AS Quarter,
    d.day_of_week AS Day_of_Week
FROM fact_air_quality AS f
INNER JOIN dim_city AS c
    ON f.city_id = c.city_id
INNER JOIN dim_date AS d
    ON f.date_id = d.date_id
INNER JOIN dim_aqi AS a
    ON f.aqi_id = a.aqi_id
ORDER BY f.measurement_id
"""

df_bd = pd.read_sql_query(consulta, conexion)
df_bd["Datetime"] = pd.to_datetime(df_bd["Datetime"])

print("Dimensiones de los datos consultados desde SQLite:", df_bd.shape)

df_bd.head()

Dimensiones de los datos consultados desde SQLite: (18265, 19)


,measurement_id,City,Datetime,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,AQI,AQI_Bucket,Year,Month,Day,Quarter,Day_of_Week
0,1,Delhi,2015-01-01,153.3,241.7,182.9,33.0,81.3,38.5,1.87,64.5,83.6,325.8,Very Poor,2015,1,1,1,Thursday
1,2,Mumbai,2015-01-01,70.5,312.7,195.0,42.0,122.5,31.5,7.22,83.8,108.0,262.7,Poor,2015,1,1,1,Thursday
2,3,Chennai,2015-01-01,174.1,275.4,56.2,68.8,230.9,28.5,8.56,60.8,43.9,341.8,Very Poor,2015,1,1,1,Thursday
3,4,Kolkata,2015-01-01,477.2,543.9,14.1,76.4,225.9,45.6,2.41,42.1,171.1,206.3,Poor,2015,1,1,1,Thursday
4,5,Bangalore,2015-01-01,171.6,117.7,123.3,12.4,61.9,49.7,1.26,79.7,164.3,339.8,Very Poor,2015,1,1,1,Thursday


In [3]:
print("Información general de los datos consultados desde SQLite:")
df_bd.info()

print("\nValores nulos por columna:")
print(df_bd.isnull().sum())

print("\nCantidad de filas duplicadas:")
print(df_bd.duplicated().sum())

Información general de los datos consultados desde SQLite:
<class 'pandas.DataFrame'>
RangeIndex: 18265 entries, 0 to 18264
Data columns (total 19 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   measurement_id  18265 non-null  int64         
 1   City            18265 non-null  str           
 2   Datetime        18265 non-null  datetime64[us]
 3   PM2.5           18265 non-null  float64       
 4   PM10            18265 non-null  float64       
 5   NO              18265 non-null  float64       
 6   NO2             18265 non-null  float64       
 7   NOx             18265 non-null  float64       
 8   NH3             18265 non-null  float64       
 9   CO              18265 non-null  float64       
 10  SO2             18265 non-null  float64       
 11  O3              18265 non-null  float64       
 12  AQI             18265 non-null  float64       
 13  AQI_Bucket      18265 non-null  str           
 14  Year  

In [4]:
variables_numericas = [
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "AQI"
]

estadisticas_descriptivas = (
    df_bd[variables_numericas]
    .describe()
    .T
)

estadisticas_descriptivas

,count,mean,std,min,25%,50%,75%,max
PM2.5,18265.0,250.597695,144.460292,0.0,125.70,251.0,376.20,499.9
PM10,18265.0,299.442491,173.479906,0.0,150.10,300.3,450.00,600.0
NO,18265.0,100.481035,57.774795,0.0,50.60,100.2,151.00,200.0
NO2,18265.0,75.415916,43.460066,0.0,37.70,76.0,113.20,150.0
NOx,18265.0,125.964079,72.403893,0.0,63.10,126.2,188.90,250.0
NH3,18265.0,25.065042,14.452019,0.0,12.60,25.3,37.60,50.0
CO,18265.0,5.002451,2.889439,0.0,2.49,5.0,7.51,10.0
SO2,18265.0,49.835839,28.988739,0.0,24.40,49.9,75.10,100.0
O3,18265.0,100.406740,57.591436,0.0,50.60,100.7,150.40,200.0
AQI,18265.0,317.505212,113.894816,26.9,207.10,331.6,414.00,500.0


In [5]:
consulta_ciudades = """
SELECT
    c.city_name AS City,
    COUNT(*) AS cantidad_registros
FROM fact_air_quality AS f
INNER JOIN dim_city AS c
    ON f.city_id = c.city_id
GROUP BY c.city_name
ORDER BY cantidad_registros DESC, City
"""

registros_por_ciudad = pd.read_sql_query(
    consulta_ciudades,
    conexion
)

print("Cantidad de registros por ciudad:")
registros_por_ciudad

consulta_categorias = """
SELECT
    a.aqi_bucket AS AQI_Bucket,
    COUNT(*) AS cantidad_registros
FROM fact_air_quality AS f
INNER JOIN dim_aqi AS a
    ON f.aqi_id = a.aqi_id
GROUP BY a.aqi_bucket
ORDER BY cantidad_registros DESC
"""

registros_por_categoria = pd.read_sql_query(
    consulta_categorias,
    conexion
)

print("Cantidad de registros por categoría AQI:")
registros_por_categoria

Cantidad de registros por ciudad:
Cantidad de registros por categoría AQI:


,AQI_Bucket,cantidad_registros
0,Very Poor,5501
1,Severe,5258
2,Moderate,4258
3,Poor,3129
4,Satisfactory,113
5,Good,6
